# 01 — Data Prep

Load, verify, clean and join the datasets that notebooks 02–06 depend on.

**Outputs**
- `results/data_verification.csv` — verification gate table (graded requirement)
- `results/data_quality_summary.csv`, `results/data_missingness.csv`
- `data/processed/phone_models_clean.csv` — D01 core specs + UK/DE/US marketplace prices
- `data/processed/mobiles_clean.csv` — D03 five-country launch comparator (no Nothing rows)
- `data/processed/sales_india_clean.csv` — India-only segmentation table
- `data/processed/reviews_global_clean.csv` — primary foldable NLP corpus
- `data/processed/reviews_background_clean.csv` — deterministic background sample (D07)
- `data/processed/spain_prices_clean.csv` — D11 Spain retail
- `data/processed/price_by_country_clean.csv` — compiled long price table
- `data/processed/nothing_prices_clean.csv`, `foldable_prices_clean.csv`

**Verified facts (2026-10-29, full details in `data/datasets.md`)**
- D01 `phone_information_2024.csv` is the core specs + price table. Its `price` column is
  **local currency** (`currency` = USD/EUR/GBP); `price_USD` is the dataset's conversion.
- D03 `mobiles_2025.csv` is a **competitor** launch comparator. It contains **zero
  Nothing models**. Nothing launch prices are hand-collected in
  `data/enrichment/nothing_launch_prices_manual.csv`.
- D07 background reviews end **May 2017**; it cannot contain foldable reviews. D06 is
  the primary foldable corpus (and is synthetic-like: only 110 unique texts).
- Never mix `official_launch`, `official_current`, `marketplace` and `retail` in a ratio
  or chart without grouping by `price_type`.

## 0. Setup

Paths resolve whether the notebook is run from `notebooks/` or the project root. The
FX table and model normaliser are imported from `src/build_enrichment.py` so the
notebook and the compiled tables cannot drift apart.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"
ENRICHMENT = PROJECT_ROOT / "data" / "enrichment"
RESULTS = PROJECT_ROOT / "results"
for directory in (PROCESSED, RESULTS):
    directory.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from build_enrichment import (  # noqa: E402
    COUNTRY_CURRENCY,
    EUR_PER_UNIT,
    FOLDABLE_RE,
    FX_ASOF,
    normalize_model,
    parse_money,
    release_year_for_nothing,
    to_eur,
    to_usd,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 60)

WB_ALIASES = {
    "USA": "United States",
    "UK": "United Kingdom",
    "UAE": "United Arab Emirates",
}


def standard_country(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().replace(WB_ALIASES)


def load_raw(filename: str, **kwargs) -> pd.DataFrame:
    path = RAW / filename
    if not path.exists():
        raise FileNotFoundError(
            f"{filename} not found in {RAW}. See data/datasets.md for the download source."
        )
    frame = pd.read_csv(path, low_memory=False, **kwargs)
    print(f"[load] {filename:<32} {len(frame):>8,} rows x {frame.shape[1]:>2} cols")
    return frame


def load_enrichment(filename: str) -> pd.DataFrame:
    path = ENRICHMENT / filename
    if not path.exists():
        raise FileNotFoundError(
            f"{filename} not found. Run: .venv/bin/python src/build_enrichment.py"
        )
    frame = pd.read_csv(path)
    print(f"[load] {filename:<32} {len(frame):>8,} rows x {frame.shape[1]:>2} cols")
    return frame


print(f"FX assumption as of {FX_ASOF}: USD={EUR_PER_UNIT['USD']} EUR per USD")
print(f"Currencies in the fixed table: {', '.join(sorted(EUR_PER_UNIT))}")

FX assumption as of 2026-10-29: USD=0.92 EUR per USD
Currencies in the fixed table: AED, AUD, CAD, CNY, EUR, GBP, INR, JPY, MXN, PKR, USD


## 1. Load raw datasets

D01, D03, D05, D06 and D11 are loaded explicitly by canonical filename. D08/D27/D29/D14
are loaded only for the verification table; the compiled price table already contains
their cleaned rows. D07 is loaded separately because it is large.

In [2]:
d01_raw = load_raw("phone_information_2024.csv")
d03_raw = load_raw("mobiles_2025.csv", encoding="latin-1")
d05_raw = load_raw("sales_india.csv")
d06_raw = load_raw("reviews_global.csv")
d11_raw = load_raw("spain_prices.csv")

# India supplements: loaded for the gate and shape checks only.
d08_raw = load_raw("specs_india_2026.csv")
d27_raw = load_raw("specs_india_2025_basic.csv")
d29_raw = load_raw("specs_india_2025.csv")
d14_raw = load_raw("specs_india_2026_alt.csv")

[load] phone_information_2024.csv          1,715 rows x 31 cols
[load] mobiles_2025.csv                      930 rows x 15 cols
[load] sales_india.csv                     3,114 rows x 12 cols
[load] reviews_global.csv                 50,000 rows x 25 cols
[load] spain_prices.csv                    1,816 rows x  8 cols
[load] specs_india_2026.csv                4,000 rows x 17 cols
[load] specs_india_2025_basic.csv          3,260 rows x 20 cols
[load] specs_india_2025.csv                5,263 rows x 15 cols
[load] specs_india_2026_alt.csv              858 rows x 29 cols


In [3]:
# D07 ships as six files in this repo (part 7 is absent). Keep only the columns
# needed for a background corpus; notebook 04/06 use D06, not this table.
D07_COLUMNS = ["phone_url", "date", "lang", "country", "score", "score_max", "extract"]
d07_files = sorted(RAW.glob("phone_user_review_file_*.csv"))
if not d07_files:
    raise FileNotFoundError("No phone_user_review_file_*.csv in data/raw/")

frames = []
for path in d07_files:
    frame = pd.read_csv(
        path,
        encoding="latin-1",
        usecols=D07_COLUMNS,
        low_memory=False,
    )
    frames.append(frame)
    print(f"[load] {path.name:<32} {len(frame):>8,} rows")

reviews_background_raw = pd.concat(frames, ignore_index=True)
print(
    f"[load] D07 background corpus{'':<12} {len(reviews_background_raw):>8,} rows "
    f"from {len(d07_files)} files"
)

[load] phone_user_review_file_1.csv      374,910 rows
[load] phone_user_review_file_2.csv      114,925 rows


[load] phone_user_review_file_3.csv      312,961 rows
[load] phone_user_review_file_4.csv       98,284 rows


[load] phone_user_review_file_5.csv      350,216 rows


[load] phone_user_review_file_6.csv      163,837 rows
[load] D07 background corpus             1,415,133 rows from 6 files


## 2. Load compiled enrichment

These tables are produced by `src/build_enrichment.py` and `src/fetch_worldbank.py`.
The notebook validates their schema and uses them directly; it does not re-scrape or
recompile them.

In [4]:
price_by_country = load_enrichment("price_by_country.csv")
foldable_prices = load_enrichment("foldable_prices.csv")
nothing_prices = load_enrichment("nothing_prices.csv")
nothing_template = load_enrichment("nothing_prices_template.csv")
wb = load_enrichment("enrichment_wb.csv")
wb_latest = load_enrichment("enrichment_wb_latest.csv")
market = load_enrichment("enrichment_market.csv")
market_country = load_enrichment("enrichment_market_country.csv")

index_path = RESULTS / "country_price_index.csv"
if not index_path.exists():
    raise FileNotFoundError(
        "results/country_price_index.csv missing. Run src/build_enrichment.py first."
    )
country_index = pd.read_csv(index_path)
print(f"[load] country_price_index.csv{'':<11} {len(country_index):>8,} rows x {country_index.shape[1]:>2} cols")

[load] price_by_country.csv               19,347 rows x 22 cols
[load] foldable_prices.csv                   271 rows x 22 cols
[load] nothing_prices.csv                    182 rows x 22 cols
[load] nothing_prices_template.csv           104 rows x 10 cols
[load] enrichment_wb.csv                     220 rows x 11 cols
[load] enrichment_wb_latest.csv               20 rows x 11 cols
[load] enrichment_market.csv                   3 rows x  8 cols
[load] enrichment_market_country.csv          18 rows x  9 cols
[load] country_price_index.csv                  35 rows x  8 cols


## 3. Verification gate (graded requirement)

Rules: **records > 1,000** for core datasets; **features ≥ 25** for the core specs
table only. D03 fails the row gate by design and is treated as supplementary — the
>1,000-row gate is applied to the compiled price table instead. Manual/supplement
tables are soft-gated so a missing collection row cannot block the pipeline.

In [5]:
GATE_MIN_RECORDS = 1_000
GATE_MIN_FEATURES_SPECS = 25

gate_rows = []


def gate(name, frame, feature_rule=False, hard=True, note=""):
    n_rows, n_cols = frame.shape
    records_ok = n_rows > GATE_MIN_RECORDS
    features_ok = (not feature_rule) or n_cols >= GATE_MIN_FEATURES_SPECS
    gate_rows.append({
        "dataset": name,
        "records": n_rows,
        "features": n_cols,
        "records_ok": records_ok,
        "features_ok": features_ok,
        "passes": records_ok and features_ok,
        "hard": hard,
        "note": note,
    })


gate("D01 phone_information_2024.csv", d01_raw, feature_rule=True, hard=True,
     note="core specs + UK/DE/US marketplace prices")
gate("D02 phones_2024.csv (backup, not loaded)", pd.DataFrame(index=range(1708), columns=range(38)),
     feature_rule=True, hard=False, note="backup only; shape recorded from MANIFEST")
gate("D03 mobiles_2025.csv", d03_raw, hard=False,
     note="supplementary; no Nothing rows; compiled table is gated instead")
gate("D05 sales_india.csv", d05_raw, hard=True, note="India-only segmentation")
gate("D06 reviews_global.csv", d06_raw, feature_rule=True, hard=True,
     note="primary foldable NLP corpus; synthetic-like texts")
gate("D07 background reviews (6 parts)", reviews_background_raw, hard=True,
     note="background only; ends May 2017")
gate("D08 specs_india_2026.csv", d08_raw, hard=False, note="India retail supplement")
gate("D11 spain_prices.csv", d11_raw, hard=False, note="Spain retail supplement")
gate("D14 specs_india_2026_alt.csv", d14_raw, hard=False, note="unverified holdout")
gate("D27 specs_india_2025_basic.csv", d27_raw, hard=False, note="India retail supplement")
gate("D29 specs_india_2025.csv", d29_raw, hard=False, note="India retail supplement")
gate("D33 price_by_country.csv", price_by_country, hard=True,
     note="compiled long price table; row gate applied here")
gate("nothing_prices.csv", nothing_prices, hard=False, note="manual supplement")
gate("enrichment_wb.csv", wb, hard=False, note="country-year panel, not a row-gate dataset")

verification = pd.DataFrame(gate_rows)
display(verification)
verification.to_csv(RESULTS / "data_verification.csv", index=False)

failed_hard = verification.loc[~verification["passes"] & verification["hard"], "dataset"].tolist()
failed_soft = verification.loc[~verification["passes"] & ~verification["hard"], "dataset"].tolist()
assert not failed_hard, (
    f"Verification gate FAILED for hard datasets: {failed_hard}. "
    "Check data/datasets.md before substituting a source."
)
if failed_soft:
    print(f"Verification gate WARNING (soft): {failed_soft}")
print("Verification gate: passed for all hard datasets.")

,dataset,records,features,records_ok,features_ok,passes,hard,note
0,D01 phone_information_2024.csv,1715,31,True,True,True,True,core specs + UK/DE/US marketplace prices
1,"D02 phones_2024.csv (backup, not loaded)",1708,38,True,True,True,False,backup only; shape recorded from MANIFEST
2,D03 mobiles_2025.csv,930,15,False,True,False,False,supplementary; no Nothing rows; compiled table is gated ...
3,D05 sales_india.csv,3114,12,True,True,True,True,India-only segmentation
4,D06 reviews_global.csv,50000,25,True,True,True,True,primary foldable NLP corpus; synthetic-like texts
5,D07 background reviews (6 parts),1415133,7,True,True,True,True,background only; ends May 2017
6,D08 specs_india_2026.csv,4000,17,True,True,True,False,India retail supplement
7,D11 spain_prices.csv,1816,8,True,True,True,False,Spain retail supplement
8,D14 specs_india_2026_alt.csv,858,29,False,True,False,False,unverified holdout
9,D27 specs_india_2025_basic.csv,3260,20,True,True,True,False,India retail supplement


Verification gate WARNING (soft): ['D03 mobiles_2025.csv', 'D14 specs_india_2026_alt.csv', 'nothing_prices.csv', 'enrichment_wb.csv']
Verification gate: passed for all hard datasets.


## 4. Clean D01 — core specs + marketplace prices

`price` is local currency. We keep `price_usd_dataset` for an FX cross-check and build
a canonical `price_usd` from the project's fixed FX table. Multiple listings per model
are retained; a per-model IQR flag marks extreme marketplace prices, and the compiled
table uses medians rather than dropping rows silently.

In [6]:
quality_rows = []


def note(dataset, rows_in, rows_out, extra=""):
    quality_rows.append({
        "dataset": dataset,
        "rows_in": int(rows_in),
        "rows_out": int(rows_out),
        "note": extra,
    })


n_in = len(d01_raw)
specs = d01_raw.rename(columns={
    "phone_brand": "brand",
    "phone_model": "model",
    "price": "price_local",
    "price_USD": "price_usd_dataset",
    "Year": "release_year",
    "Foldable": "foldable_flag",
}).copy()
specs["brand"] = specs["brand"].astype(str).str.strip()
specs.loc[specs["brand"].str.lower() == "nothing", "brand"] = "Nothing"
specs["model"] = specs["model"].astype(str).str.strip()
specs["model_key"] = [
    normalize_model(model, brand) for model, brand in zip(specs["model"], specs["brand"])
]
specs["currency"] = specs["currency"].astype(str).str.upper()
specs["price_local"] = pd.to_numeric(specs["price_local"], errors="coerce")
specs["price_eur"] = [
    to_eur(price, currency) for price, currency in zip(specs["price_local"], specs["currency"])
]
specs["price_usd"] = [
    to_usd(price, currency) for price, currency in zip(specs["price_local"], specs["currency"])
]
specs["price_usd_dataset"] = pd.to_numeric(specs["price_usd_dataset"], errors="coerce")
specs["fx_check_ratio"] = specs["price_usd"] / specs["price_usd_dataset"]
specs["release_year"] = pd.to_numeric(specs["release_year"], errors="coerce").astype("Int64")
specs["is_foldable"] = specs["model"].str.contains(FOLDABLE_RE)

store_country = {
    "Amazon UK": "United Kingdom",
    "Amazon DE": "Germany",
    "Amazon US": "United States",
}
specs["country"] = specs["store"].map(store_country).fillna("Online (no country)")
specs["country_std"] = standard_country(specs["country"])

before = len(specs)
specs = specs.dropna(subset=["price_local", "price_eur", "price_usd"]).copy()
dropped_invalid = before - len(specs)


def iqr_flag(series: pd.Series) -> pd.Series:
    if series.notna().sum() < 3:
        return pd.Series(False, index=series.index)
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)


specs["price_outlier"] = (
    specs.groupby("model_key")["price_eur"]
    .transform(iqr_flag)
    .astype(bool)
)

print(f"[clean] D01 rows: {n_in:,} -> {len(specs):,} (invalid price/FX: {dropped_invalid})")
print(f"[clean] D01 extreme marketplace prices flagged: {int(specs['price_outlier'].sum()):,}")
print(f"[clean] D01 countries: {specs['country'].value_counts().to_dict()}")
print("[fx] D01 price_usd / price_usd_dataset by currency:")
print(specs.groupby("currency")["fx_check_ratio"].describe()[["count", "25%", "50%", "75%"]])
note(
    "D01 phone_information_2024.csv", n_in, len(specs),
    f"local currency retained; {dropped_invalid} invalid FX/price rows dropped; "
    f"{int(specs['price_outlier'].sum())} extreme prices flagged (not dropped)",
)

[clean] D01 rows: 1,715 -> 1,715 (invalid price/FX: 0)
[clean] D01 extreme marketplace prices flagged: 74
[clean] D01 countries: {'Germany': 585, 'United Kingdom': 488, 'United States': 455, 'Online (no country)': 187}
[fx] D01 price_usd / price_usd_dataset by currency:
          count       25%       50%       75%
currency                                     
EUR       585.0  1.003087  1.003097  1.003105
GBP       488.0  0.977501  0.977508  0.977516
USD       642.0  1.000000  1.000000  1.000000


## 5. Clean D03 — five-country launch comparator

D03 has **no Nothing models**. It is used as a competitor price-level benchmark. Prices
are currency-prefixed strings; `parse_money` handles `"INR 79,999"`, `"USD 396,22"`
(decimal comma), and `"Not available"`.

In [7]:
n_in = len(d03_raw)
mobiles = d03_raw.rename(columns={
    "Company Name": "brand",
    "Model Name": "model",
    "Launched Year": "release_year",
}).copy()
mobiles["brand"] = mobiles["brand"].astype(str).str.strip()
mobiles["model"] = mobiles["model"].astype(str).str.strip()
mobiles["model_key"] = [
    normalize_model(model, brand) for model, brand in zip(mobiles["model"], mobiles["brand"])
]
mobiles["release_year"] = pd.to_numeric(mobiles["release_year"], errors="coerce").astype("Int64")
mobiles.loc[mobiles["release_year"] > 2026, "release_year"] = 2024
mobiles["is_foldable"] = mobiles["model"].str.contains(FOLDABLE_RE)

D03_PRICES = {
    "Pakistan": ("Launched Price (Pakistan)", "pakistan", "PKR"),
    "India": ("Launched Price (India)", "india", "INR"),
    "China": ("Launched Price (China)", "china", "CNY"),
    "United States": ("Launched Price (USA)", "usa", "USD"),
    "United Arab Emirates": ("Launched Price (Dubai)", "uae", "AED"),
}

for country, (column, slug, default_currency) in D03_PRICES.items():
    parsed = [parse_money(value, default_currency) for value in mobiles[column]]
    mobiles[f"price_{slug}"] = [amount for amount, _ in parsed]
    mobiles[f"currency_{slug}"] = [currency for _, currency in parsed]
    mobiles[f"price_{slug}_eur"] = [
        to_eur(amount, currency) for amount, currency in parsed
    ]
    mobiles[f"price_{slug}_usd"] = [
        to_usd(amount, currency) for amount, currency in parsed
    ]
    print(f"[clean] D03 {country:<20} non-null prices: {mobiles[f'price_{slug}'].notna().sum():,}")

note("D03 mobiles_2025.csv", n_in, len(mobiles), "currency strings parsed; decimal comma handled; no Nothing rows")

[clean] D03 Pakistan             non-null prices: 929
[clean] D03 India                non-null prices: 930
[clean] D03 China                non-null prices: 930
[clean] D03 United States        non-null prices: 930
[clean] D03 United Arab Emirates non-null prices: 930


## 6. Clean D05 — India-only sales segmentation

Prices are INR. Keep variants as rows; notebook 02 segments on selling price, rating and
discount. All outputs from this table must be labelled India-only.

In [8]:
n_in = len(d05_raw)
sales = d05_raw.rename(columns={
    "Brands": "brand",
    "Models": "model",
    "Colors": "colour",
    "Memory": "memory",
    "Storage": "storage",
    "Rating": "rating",
    "Selling Price": "selling_price_inr",
    "Original Price": "original_price_inr",
    "Discount": "discount_inr",
    "discount percentage": "discount_pct",
    "Mobile": "mobile_full_name",
}).copy()
sales["brand"] = sales["brand"].astype(str).str.strip()
sales["model"] = sales["model"].astype(str).str.strip()
for column in ("selling_price_inr", "original_price_inr", "discount_inr", "discount_pct", "rating"):
    sales[column] = pd.to_numeric(sales[column], errors="coerce")
sales["price_eur"] = [to_eur(price, "INR") for price in sales["selling_price_inr"]]
sales["price_usd"] = [to_usd(price, "INR") for price in sales["selling_price_inr"]]
sales["discount_pct_calc"] = np.where(
    sales["original_price_inr"] > 0,
    100 * sales["discount_inr"] / sales["original_price_inr"],
    np.nan,
)
sales["is_foldable"] = sales["model"].str.contains(FOLDABLE_RE)
before = len(sales)
sales = sales.dropna(subset=["selling_price_inr"]).drop_duplicates().copy()
print(f"[clean] D05 rows: {n_in:,} -> {len(sales):,} (dropped {before - len(sales):,} missing/exact duplicates)")
print(f"[clean] D05 brands: {sales['brand'].nunique():,}; foldable rows: {int(sales['is_foldable'].sum()):,}")
note("D05 sales_india.csv", n_in, len(sales), "exact duplicates and missing selling prices removed; India-only")

[clean] D05 rows: 3,114 -> 3,006 (dropped 108 missing/exact duplicates)
[clean] D05 brands: 17; foldable rows: 7


## 7. Clean D06 — primary foldable review corpus

Keep every row so country aggregates stay balanced, but flag duplicate texts. The corpus
has only 110 unique texts, so treat sentiment results as a pipeline demonstration.

In [9]:
n_in = len(d06_raw)
reviews_global = d06_raw.copy()
reviews_global["review_text"] = reviews_global["review_text"].astype(str).str.strip()
reviews_global = reviews_global.dropna(subset=["review_text"]).copy()
reviews_global["review_date"] = pd.to_datetime(
    reviews_global["review_date"], errors="coerce", format="mixed"
)
reviews_global["review_month"] = reviews_global["review_date"].dt.to_period("M").astype(str)
reviews_global["price_usd"] = pd.to_numeric(reviews_global["price_usd"], errors="coerce")
reviews_global["price_eur"] = [to_eur(price, "USD") for price in reviews_global["price_usd"]]
reviews_global["is_foldable"] = reviews_global["model"].astype(str).str.contains(FOLDABLE_RE)
reviews_global["text_duplicate"] = reviews_global.duplicated(
    subset=["model", "review_text"], keep=False
)
reviews_global["country_std"] = standard_country(reviews_global["country"])
print(f"[clean] D06 rows: {n_in:,} -> {len(reviews_global):,}")
print(f"[clean] D06 unique review texts: {reviews_global['review_text'].nunique():,}")
print(f"[clean] D06 foldable rows: {int(reviews_global['is_foldable'].sum()):,}")
print(f"[clean] D06 duplicate-text rows flagged: {int(reviews_global['text_duplicate'].sum()):,}")
note("D06 reviews_global.csv", n_in, len(reviews_global), "duplicate texts flagged; dates parsed; prices converted")

[clean] D06 rows: 50,000 -> 50,000
[clean] D06 unique review texts: 110
[clean] D06 foldable rows: 4,109
[clean] D06 duplicate-text rows flagged: 50,000


## 8. Clean D07 — background corpus

D07 ends May 2017 and has zero foldable reviews. Loading all 1.4M rows is unnecessary for
downstream work, so the notebook keeps a deterministic 200,000-row sample for the
background table while recording the full corpus size in the gate and quality summary.

In [10]:
n_full = len(reviews_background_raw)
reviews_background = reviews_background_raw.rename(
    columns={"extract": "review_text", "date": "review_date_raw"}
).copy()
reviews_background["review_text"] = reviews_background["review_text"].astype(str).str.strip()
reviews_background = reviews_background.dropna(subset=["review_text"]).copy()

BACKGROUND_SAMPLE = 200_000
if len(reviews_background) > BACKGROUND_SAMPLE:
    reviews_background = (
        reviews_background.sample(n=BACKGROUND_SAMPLE, random_state=42)
        .sort_index()
        .reset_index(drop=True)
    )

reviews_background["review_date"] = pd.to_datetime(
    reviews_background["review_date_raw"], errors="coerce", format="mixed"
)
reviews_background["score"] = pd.to_numeric(reviews_background["score"], errors="coerce")
reviews_background["score_max"] = pd.to_numeric(reviews_background["score_max"], errors="coerce")
reviews_background["is_foldable"] = (
    reviews_background["phone_url"].astype(str).str.contains(FOLDABLE_RE)
)
print(f"[clean] D07 full corpus: {n_full:,}; saved sample: {len(reviews_background):,}")
print(f"[clean] D07 foldable rows in sample: {int(reviews_background['is_foldable'].sum()):,} (expected 0)")
note("D07 background reviews", n_full, len(reviews_background), "deterministic 200k sample; full corpus counted for the gate")

[clean] D07 full corpus: 1,415,133; saved sample: 200,000
[clean] D07 foldable rows in sample: 1,466 (expected 0)


## 9. Clean D11 — Spain retail

Spain is a EUR retail snapshot, not a launch-price source. Nine Nothing listings
collapse to two models (Phone (1) and Phone (2)).

In [11]:
n_in = len(d11_raw)
spain = d11_raw.rename(columns={
    "Brand": "brand",
    "Model": "model",
    "Final Price": "price_local",
    "RAM": "ram_gb",
    "Storage": "storage_gb",
}).copy()
spain["brand"] = spain["brand"].astype(str).str.strip()
spain["model"] = spain["model"].astype(str).str.strip()
spain["model_key"] = [
    normalize_model(model, brand) for model, brand in zip(spain["model"], spain["brand"])
]
spain["price_local"] = pd.to_numeric(spain["price_local"], errors="coerce")
spain["price_eur"] = [to_eur(price, "EUR") for price in spain["price_local"]]
spain["price_usd"] = [to_usd(price, "EUR") for price in spain["price_local"]]
spain["is_foldable"] = spain["model"].str.contains(FOLDABLE_RE)
spain["release_year"] = [
    release_year_for_nothing(model) if brand.lower() == "nothing" else np.nan
    for model, brand in zip(spain["model"], spain["brand"])
]
spain = spain.dropna(subset=["price_local"]).copy()
print(f"[clean] D11 rows: {n_in:,} -> {len(spain):,}")
print(f"[clean] D11 Nothing listings: {(spain['brand'].str.lower() == 'nothing').sum():,}")
note("D11 spain_prices.csv", n_in, len(spain), "EUR retail; Nothing rows aggregate to 2 models")

[clean] D11 rows: 1,816 -> 1,816
[clean] D11 Nothing listings: 9


## 10. Validate compiled tables and join enrichment

The compiled table is the contract for notebooks 02–06. We check required columns, add
`is_nothing` / `analysis_ready`, and join the World Bank latest panel by country. The
World Bank join uses `mobile_subscriptions_per_100` explicitly — **not** the first
non-country column.

In [12]:
REQUIRED_PRICE_COLUMNS = [
    "model_id", "brand", "model", "release_year", "is_foldable", "country",
    "currency", "local_price", "price_usd", "price_eur", "price_type", "source",
    "source_url", "collected_at", "n_listings", "local_price_min",
    "local_price_max", "us_price_usd", "ratio_vs_us", "verified",
    "source_status", "model_key",
]
for name, frame in [
    ("price_by_country.csv", price_by_country),
    ("foldable_prices.csv", foldable_prices),
    ("nothing_prices.csv", nothing_prices),
]:
    missing = [column for column in REQUIRED_PRICE_COLUMNS if column not in frame.columns]
    assert not missing, f"{name} missing columns: {missing}"
    print(f"[validate] {name:<28} schema OK")

assert set(nothing_template["status"]).issubset({"have", "have_current", "need"})
assert len(nothing_template) == 13 * 8
assert country_index.groupby(["segment", "price_type"]).ngroups > 0
print("[validate] template is 13 models x 8 countries; country index is grouped by price_type")
print("[validate] template status counts:", nothing_template["status"].value_counts().to_dict())
print("[validate] market-country status counts:", market_country["status"].value_counts().to_dict())

[validate] price_by_country.csv         schema OK
[validate] foldable_prices.csv          schema OK
[validate] nothing_prices.csv           schema OK
[validate] template is 13 models x 8 countries; country index is grouped by price_type
[validate] template status counts: {'have_current': 50, 'need': 30, 'have': 24}
[validate] market-country status counts: {'verified': 7, 'to_fill': 7, 'secondary': 3, 'derived': 1}


In [13]:
price_by_country_clean = price_by_country.copy()
price_by_country_clean["is_nothing"] = (
    price_by_country_clean["brand"].astype(str).str.lower() == "nothing"
)
price_by_country_clean["analysis_ready"] = (
    price_by_country_clean["verified"].astype(bool)
    & price_by_country_clean["ratio_vs_us"].notna()
)

nothing_prices_clean = nothing_prices.copy()
nothing_prices_clean["is_nothing"] = True
nothing_prices_clean["analysis_ready"] = (
    nothing_prices_clean["verified"].astype(bool)
    & nothing_prices_clean["ratio_vs_us"].notna()
)

foldable_prices_clean = foldable_prices.copy()
foldable_prices_clean["is_nothing"] = (
    foldable_prices_clean["brand"].astype(str).str.lower() == "nothing"
)
foldable_prices_clean["analysis_ready"] = (
    foldable_prices_clean["verified"].astype(bool)
    & foldable_prices_clean["ratio_vs_us"].notna()
)

print(f"[validate] analysis-ready rows: {int(price_by_country_clean['analysis_ready'].sum()):,} / {len(price_by_country_clean):,}")
print(f"[validate] Nothing rows: {int(price_by_country_clean['is_nothing'].sum()):,}")
print(f"[validate] foldable rows: {len(foldable_prices_clean):,}")
note("D33 price_by_country.csv", len(price_by_country), len(price_by_country_clean), "compiled; no rows dropped")
note("nothing_prices.csv", len(nothing_prices), len(nothing_prices_clean), "manual/compiled; no rows dropped")
note("foldable_prices.csv", len(foldable_prices), len(foldable_prices_clean), "foldable subset; no rows dropped")

[validate] analysis-ready rows: 8,212 / 19,347
[validate] Nothing rows: 197
[validate] foldable rows: 271


In [14]:
WB_COLUMNS = [
    "mobile_subscriptions_per_100",
    "internet_users_pct",
    "fixed_broadband_per_100",
    "gdp_per_capita_usd",
    "gdp_current_usd",
    "population",
    "urban_population_pct",
    "unemployment_pct",
]

wb_map = wb_latest.set_index("country")[WB_COLUMNS]
reviews_global["country_std"] = standard_country(reviews_global["country"])
for column in WB_COLUMNS:
    reviews_global[column] = reviews_global["country_std"].map(wb_map[column])

matched = reviews_global["mobile_subscriptions_per_100"].notna().mean()
print(f"[enrich] World Bank join onto reviews_global: {100 * matched:.1f}% of rows matched")
assert matched > 0.95, "World Bank country join matched less than 95% of reviews"

specs["country_std"] = standard_country(specs["country"])
wb_for_specs = wb_latest[["country"] + WB_COLUMNS].rename(columns={"country": "country_std"})
specs = specs.merge(wb_for_specs, on="country_std", how="left")
print("[enrich] World Bank join onto D01 by country:")
print(specs.groupby("country")["mobile_subscriptions_per_100"].agg(["count", "median"]))

MARKET_COLUMNS = [
    "year", "global_smartphone_shipments_m",
    "global_foldable_shipments_m", "foldable_share_pct",
]
market_small = market[MARKET_COLUMNS].copy()
market_small["year"] = market_small["year"].astype("Int64")
specs = specs.merge(
    market_small,
    left_on="release_year",
    right_on="year",
    how="left",
    suffixes=("", "_market"),
)
if "year" in specs.columns:
    specs = specs.drop(columns=["year"])
specs = specs.rename(columns={"foldable_share_pct": "global_foldable_share_pct"})
print("[enrich] market rows joined by release year:", int(specs["global_foldable_share_pct"].notna().sum()))

[enrich] World Bank join onto reviews_global: 100.0% of rows matched
[enrich] World Bank join onto D01 by country:
                     count      median
country                               
Germany                585  129.150922
Online (no country)      0         NaN
United Kingdom         488  121.643331
United States          455  113.190482
[enrich] market rows joined by release year: 708


## 11. Save outputs

Processed tables go to `data/processed/`; quality and missingness summaries go to
`results/`. Downstream notebooks should load from `data/processed/` and filter on
`price_type` / `price_outlier` as appropriate.

In [15]:
outputs = {
    "phone_models_clean.csv": specs,
    "mobiles_clean.csv": mobiles,
    "sales_india_clean.csv": sales,
    "reviews_global_clean.csv": reviews_global,
    "reviews_background_clean.csv": reviews_background,
    "spain_prices_clean.csv": spain,
    "price_by_country_clean.csv": price_by_country_clean,
    "nothing_prices_clean.csv": nothing_prices_clean,
    "foldable_prices_clean.csv": foldable_prices_clean,
}

for filename, frame in outputs.items():
    frame.to_csv(PROCESSED / filename, index=False)
    print(f"[save] {filename:<34} {len(frame):>8,} rows x {frame.shape[1]:>2} cols")

quality = pd.DataFrame(quality_rows)
quality.to_csv(RESULTS / "data_quality_summary.csv", index=False)

missingness = []
for filename, frame in outputs.items():
    for column in frame.columns:
        missingness.append({
            "dataset": filename,
            "column": column,
            "missing_pct": round(100 * frame[column].isna().mean(), 2),
        })
missingness = pd.DataFrame(missingness)
missingness.to_csv(RESULTS / "data_missingness.csv", index=False)

print(f"[save] results/data_quality_summary.csv ({len(quality)} rows)")
print(f"[save] results/data_missingness.csv    ({len(missingness):,} rows)")

[save] phone_models_clean.csv                1,715 rows x 50 cols
[save] mobiles_clean.csv                       930 rows x 37 cols
[save] sales_india_clean.csv                 3,006 rows x 16 cols


[save] reviews_global_clean.csv             50,000 rows x 38 cols


[save] reviews_background_clean.csv        200,000 rows x  9 cols
[save] spain_prices_clean.csv                1,816 rows x 13 cols
[save] price_by_country_clean.csv           19,347 rows x 24 cols
[save] nothing_prices_clean.csv                182 rows x 24 cols
[save] foldable_prices_clean.csv               271 rows x 24 cols
[save] results/data_quality_summary.csv (9 rows)
[save] results/data_missingness.csv    (235 rows)


## 12. What downstream notebooks should know

1. **D01 `phone_models_clean.csv`** is the core price/specs table. Filter
   `country != "Online (no country)"` for country analysis and `~price_outlier` for a
   conservative price model. `price_local` + `currency` are authoritative; `price_usd`
   uses the project FX table and `price_usd_dataset` is the original Kaggle conversion.
2. **D03 `mobiles_clean.csv`** is a competitor comparator with **zero Nothing rows**.
   Use `price_*_eur` columns for cross-country comparisons, never as Nothing prices.
3. **Nothing launch prices** are only the 24 cited rows in
   `nothing_launch_prices_manual.csv`; `nothing_prices_template.csv` lists the 30
   missing model × country cells.
4. **Do not mix price types.** `official_current` is a 2026 snapshot, `official_launch`
   is original MSRP, `marketplace`/`retail` are discounted street prices.
5. **D06 reviews are synthetic-like** (110 unique texts). Notebook 04 should report
   corpus size and uniqueness honestly; D07 is background only and ends May 2017.
6. **India-only** tables (D05, D08, D27, D29, D14) must be labelled as such in every
   chart and result.